In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
import os

os.makedirs("../outputs/figures", exist_ok=True)
sns.set_theme(style="whitegrid", palette="muted")


In [2]:

# Load clean data

df = pd.read_csv("../data/clean/msme_survey_clean.csv")

print("ANALYSIS - MSME IMPACT EVALUATION")
print(f"Total observations loaded: {len(df)}")

df_analysis = df[df["endline_revenue_missing"] == 0].copy()
print(f"Observations with endline revenue (used in analysis): {len(df_analysis)}")
print(f"Observations dropped (missing endline): {len(df) - len(df_analysis)}")




ANALYSIS - MSME IMPACT EVALUATION
Total observations loaded: 500
Observations with endline revenue (used in analysis): 478
Observations dropped (missing endline): 22


In [3]:
# STEP 1 - Descriptive Statistics

print("STEP 1: DESCRIPTIVE STATISTICS BY GROUP")
print("=" * 55)

desc_vars = [
    "baseline_revenue_inr",
    "endline_revenue_inr",
    "employees_baseline",
    "loan_access_baseline",
    "business_age_years"
]

summary = df_analysis.groupby("treatment")[desc_vars].agg(["mean", "std"]).round(2)
print(summary)
summary.to_csv("../outputs/summary_table.csv")
print("\nSummary table saved to: outputs/summary_table.csv")




STEP 1: DESCRIPTIVE STATISTICS BY GROUP
          baseline_revenue_inr           endline_revenue_inr            \
                          mean       std                mean       std   
treatment                                                                
0                     49190.63  44225.05            49753.53  46121.69   
1                     50264.57  48738.25            57667.20  56466.01   

          employees_baseline       loan_access_baseline        \
                        mean   std                 mean   std   
treatment                                                       
0                       7.18  4.13                 0.40  0.49   
1                       7.50  4.09                 0.35  0.48   

          business_age_years        
                        mean   std  
treatment                           
0                      10.59  5.34  
1                      10.28  5.51  

Summary table saved to: outputs/summary_table.csv


In [4]:
# STEP 2 - Balance Check
# t-tests on baseline variables to verify randomization worked

print("STEP 2: BALANCE CHECK (t-tests on baseline variables)")
print("=" * 55)

balance_vars = [
    "baseline_revenue_inr",
    "employees_baseline",
    "loan_access_baseline",
    "business_age_years"
]

balance_results = []

for var in balance_vars:
    treatment_vals = df_analysis[df_analysis["treatment"] == 1][var].dropna()
    control_vals   = df_analysis[df_analysis["treatment"] == 0][var].dropna()
    t_stat, p_value = stats.ttest_ind(treatment_vals, control_vals)

    balance_results.append({
        "Variable":        var,
        "Treatment Mean":  round(treatment_vals.mean(), 2),
        "Control Mean":    round(control_vals.mean(), 2),
        "t-statistic":     round(t_stat, 3),
        "p-value":         round(p_value, 3),
        "Balanced?":       "Yes" if p_value > 0.05 else "No (p < 0.05)"
    })

balance_df = pd.DataFrame(balance_results)
print(balance_df.to_string(index=False))




STEP 2: BALANCE CHECK (t-tests on baseline variables)
            Variable  Treatment Mean  Control Mean  t-statistic  p-value Balanced?
baseline_revenue_inr        50264.57      49190.63        0.242    0.809       Yes
  employees_baseline            7.50          7.18        0.830    0.407       Yes
loan_access_baseline            0.35          0.40       -1.017    0.310       Yes
  business_age_years           10.28         10.59       -0.600    0.549       Yes


In [5]:
# STEP 3 - OLS Regression (using sklearn)
# Estimating the effect of treatment on log endline revenue,
# controlling for baseline revenue, loan access, and business age.
# 
# Model:
#   log(endline_revenue) = β0 + β1*treatment + β2*log(baseline_revenue)
#                        + β3*loan_access + β4*business_age + state_dummies
#
# β1 is the main coefficient - the estimated treatment effect.

print("STEP 3: OLS REGRESSION - TREATMENT EFFECT ON REVENUE")

# Drop rows with any missing values in the variables we need
reg_vars = [
    "log_endline_revenue",
    "treatment",
    "log_baseline_revenue",
    "loan_access_baseline",
    "business_age_years"
]
state_dummy_cols = [c for c in df_analysis.columns if c.startswith("state_")]
reg_vars += state_dummy_cols

df_reg = df_analysis[reg_vars].dropna().copy()
print(f"Observations used in regression: {len(df_reg)}")

# Separate outcome (y) and predictors (X)
y = df_reg["log_endline_revenue"].values
X = df_reg.drop(columns=["log_endline_revenue"]).values.astype(float)
feature_names = df_reg.drop(columns=["log_endline_revenue"]).columns.tolist()

# Fit OLS using sklearn
model = LinearRegression()
model.fit(X, y)

# Compute R-squared
y_pred = model.predict(X)
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r_squared = 1 - (ss_res / ss_tot)

# Compute standard errors manually
n = len(y)
k = X.shape[1]
residuals = y - y_pred
sigma_sq = ss_res / (n - k - 1)                         # residual variance
X_with_intercept = np.column_stack([np.ones(n), X])
cov_matrix = sigma_sq * np.linalg.pinv(X_with_intercept.T @ X_with_intercept)
std_errors = np.sqrt(np.diag(cov_matrix))[1:]           # skip intercept SE
t_stats = model.coef_ / std_errors
p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n - k - 1))

# Print results table for main variables
print(f"\n{'Variable':<28} {'Coef':>8} {'Std Err':>9} {'t':>8} {'p-value':>9}")
main_vars = ["treatment", "log_baseline_revenue", "loan_access_baseline", "business_age_years"]
for name, coef, se, t, p in zip(feature_names, model.coef_, std_errors, t_stats, p_values):
    if name in main_vars:
        sig = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
        print(f"{name:<28} {coef:>8.4f} {se:>9.4f} {t:>8.3f} {p:>9.4f} {sig}")

print(f"R-squared: {r_squared:.4f}")
print("Significance: *** p<0.01  ** p<0.05  * p<0.1")

# Interpret the treatment coefficient
treatment_idx = feature_names.index("treatment")
treatment_coef = model.coef_[treatment_idx]
treatment_pval = p_values[treatment_idx]

print(f"\nKey result:")
print(f"  Treatment coefficient: {treatment_coef:.4f}")
print(f"  p-value:               {treatment_pval:.4f}")

if treatment_pval < 0.05:
    pct_effect = (np.exp(treatment_coef) - 1) * 100
    print(f"  Interpretation: Treatment is associated with a "
          f"{pct_effect:.1f}% increase in endline revenue (statistically significant)")
else:
    print(f"  Interpretation: Treatment effect is not statistically "
          f"significant at the 5% level")




STEP 3: OLS REGRESSION - TREATMENT EFFECT ON REVENUE
Observations used in regression: 478

Variable                         Coef   Std Err        t   p-value
treatment                      0.1265    0.0091   13.864    0.0000 ***
log_baseline_revenue           1.0018    0.0055  183.518    0.0000 ***
loan_access_baseline          -0.0159    0.0092   -1.730    0.0843 *
business_age_years             0.0008    0.0008    1.019    0.3088 
R-squared: 0.9866
Significance: *** p<0.01  ** p<0.05  * p<0.1

Key result:
  Treatment coefficient: 0.1265
  p-value:               0.0000
  Interpretation: Treatment is associated with a 13.5% increase in endline revenue (statistically significant)


In [6]:
# STEP 4 - Visualizations

print("STEP 4: GENERATING CHARTS")


# Chart 1: Revenue Distribution - Treatment vs Control
fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    data=df_analysis,
    x="log_endline_revenue",
    hue="treatment",
    bins=30,
    kde=True,
    palette={1: "#2196F3", 0: "#FF7043"},
    alpha=0.6,
    ax=ax
)

ax.set_title("Distribution of Endline Revenue\n(Treatment vs Control)", fontsize=13)
ax.set_xlabel("Log Endline Revenue (INR)", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
handles, _ = ax.get_legend().legend_handles, ax.get_legend().get_texts()
ax.legend(handles, ["Control", "Treatment"], title="Group")

plt.tight_layout()
plt.savefig("../outputs/figures/01_revenue_distribution.png", dpi=150)
plt.close()
print("  Chart 1 saved: 01_revenue_distribution.png")


# Chart 2: Balance Check Bar Chart
fig, ax = plt.subplots(figsize=(9, 5))

balance_df["Diff (%)"] = (
    (balance_df["Treatment Mean"] - balance_df["Control Mean"])
    / balance_df["Control Mean"] * 100
).round(2)

colors = ["#4CAF50" if b == "Yes" else "#F44336" for b in balance_df["Balanced?"]]

ax.barh(balance_df["Variable"], balance_df["Diff (%)"], color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Balance Check: % Difference in Baseline Variables\n(Treatment vs Control)",
             fontsize=13)
ax.set_xlabel("% Difference (Treatment − Control)", fontsize=11)

for i, row in balance_df.iterrows():
    ax.text(row["Diff (%)"] + 0.1, i, f"p={row['p-value']}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("../outputs/figures/02_balance_check.png", dpi=150)
plt.close()
print("  Chart 2 saved: 02_balance_check.png")


# Chart 3: Regression Coefficient Plot (main variables only)
fig, ax = plt.subplots(figsize=(8, 5))

coef_data = {
    "Treatment":              (model.coef_[feature_names.index("treatment")],
                               std_errors[feature_names.index("treatment")]),
    "Log Baseline Revenue":   (model.coef_[feature_names.index("log_baseline_revenue")],
                               std_errors[feature_names.index("log_baseline_revenue")]),
    "Loan Access (Baseline)": (model.coef_[feature_names.index("loan_access_baseline")],
                               std_errors[feature_names.index("loan_access_baseline")]),
    "Business Age (Years)":   (model.coef_[feature_names.index("business_age_years")],
                               std_errors[feature_names.index("business_age_years")]),
}

labels = list(coef_data.keys())
coefs  = [v[0] for v in coef_data.values()]
errors = [v[1] * 1.96 for v in coef_data.values()]      # 95% CI
colors = ["#2196F3" if c > 0 else "#FF7043" for c in coefs]

ax.barh(labels, coefs, xerr=errors, color=colors, edgecolor="white", capsize=4)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("OLS Regression Coefficients\n(Dependent Variable: Log Endline Revenue)",
             fontsize=13)
ax.set_xlabel("Coefficient (with 95% CI)", fontsize=11)

plt.tight_layout()
plt.savefig("../outputs/figures/03_regression_coefficients.png", dpi=150)
plt.close()
print("Chart 3 saved: 03_regression_coefficients.png")

# Done

print("ANALYSIS COMPLETE")
print("Outputs saved to:")
print("../outputs/summary_table.csv")
print("../outputs/figures/01_revenue_distribution.png")
print("../outputs/figures/02_balance_check.png")
print("../outputs/figures/03_regression_coefficients.png")

STEP 4: GENERATING CHARTS
  Chart 1 saved: 01_revenue_distribution.png
  Chart 2 saved: 02_balance_check.png
Chart 3 saved: 03_regression_coefficients.png
ANALYSIS COMPLETE
Outputs saved to:
../outputs/summary_table.csv
../outputs/figures/01_revenue_distribution.png
../outputs/figures/02_balance_check.png
../outputs/figures/03_regression_coefficients.png
